# Intro to Machine Learning

[Course Link](https://www.kaggle.com/learn/intermediate-machine-learning)

In [48]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [17]:
CWD = os.getcwd()
DATA = CWD + '/tutorials_data/'

TRAIN = DATA + 'housing_train.csv'
TEST = DATA + 'housing_test.csv'
MELB = DATA + 'housing_melbourne.csv'

## 1. Introduction

In this course, you will learn to:
- Tackle data types in real-world datasets (missing values, categorical variables)
- Design pipelines to improve quality of machine learning code
- Use advanced techniques for model validation (cross-validation)
- Build models used in competitions (XGBoost)
- Avoid common data science mistakes (leakage)

### Format Data

In [7]:
# Load data
df_train = pd.read_csv(TRAIN)
df_test = pd.read_csv(TEST)

# Extract target
y = df_train['SalePrice']
# Define features
features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']
# Extract features
X = df_train[features].copy()
# Extract test feature data
X_test = df_test[features].copy()

# Split training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state = 0)

# Show data
X_train.head()

,LotArea,YearBuilt,1stFlrSF,2ndFlrSF,FullBath,BedroomAbvGr,TotRmsAbvGrd
618,11694,2007,1828,0,2,3,9
870,6600,1962,894,0,1,2,5
92,13360,1921,964,0,1,2,5
817,13265,2002,1689,0,2,3,7
302,13704,2001,1541,0,2,3,6


### Define Models

In [8]:
# Define multiple RandomForest models
model_1 = RandomForestRegressor(n_estimators = 50, random_state = 0)
model_2 = RandomForestRegressor(n_estimators = 100, random_state = 0)
model_3 = RandomForestRegressor(n_estimators = 100, random_state = 0, criterion = 'absolute_error')
model_4 = RandomForestRegressor(n_estimators = 200, random_state = 0, min_samples_split = 20)
model_5 = RandomForestRegressor(n_estimators = 100, random_state = 0, max_depth = 7) 

# Define model list
list_models = [model_1, model_2, model_3, model_4, model_5]

### Define Scoring Function

In [10]:
def score_model(model, X_train = X_train, X_valid = X_valid, y_train = y_train, y_valid = y_valid):
    # Fit model
    model.fit(X_train, y_train)
    # Generate predictions
    y_predictions = model.predict(X_valid)
    
    return mean_absolute_error(y_valid, y_predictions)

### Test Models

In [16]:
# Initialise best model
best_mae = np.inf
best_model = None

# Iterate over model
for i in range(0, len(list_models)):
    # Calculate MAE
    mae = score_model(list_models[i])
    # Check for best MAE
    if mae < best_mae:
        # Assign best_mae
        best_mae = mae
        # Assign best_model
        best_model = i+1
    print(f'Model {i + 1} MAE: {mae}')

print()
print(f'Best Model: {best_model}')

Model 1 MAE: 24015.492818003917
Model 2 MAE: 23740.979228636657
Model 3 MAE: 23528.78421232877
Model 4 MAE: 23996.676789668687
Model 5 MAE: 23706.672864217904

Best Model: 3


## 2. Missing Values

There are many ways data ends up with missing values:
- A 2 bedroom house won't include value for size of a third bedroom
- A respondent may choose not to share income data

Most ML libraries, including `scikit-learn` give an error when building a model with missing values in the data. 

There are various strategies to deal with missing data.

### Drop Columns

The simplest option is to drop feature columns with missing values.

Unless most values in the dropped column are missing, them odel loses access to a lot of potentially relevant information.

### Imputation

Imputation fills missing values with a calculated value, often the **mean** of a column. 

This value won't be strictly correct in most cases; however, it leads to more accurate models than dropping columns entirely.

### Extended Imputation

Simple imputed values may be systematically above or below actual values, or rows including missing values may be unique in some other way.

In this case, the model would make better predictions by considering which values were originally missing.

In this approach, the missing values are imputed for a given column, and an additional column is created which shows the location of those imputed values.

### Example

In this example, these three methods will be tested on the Melbourne Dataset.

In [18]:
# Load data
df = pd.read_csv(MELB)

# Select target
y = df['Price']

# Select numerical features
features = df.drop(['Price'], axis = 1)
X = features.select_dtypes(exclude = ['object'])

# Divide data into train/test/validation
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state = 0)

# Show data
X_train.head()

,Rooms,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
12167,1,5.0,3182.0,1.0,1.0,1.0,0.0,NaN,1940.0,-37.85984,144.9867,13240.0
6524,2,8.0,3016.0,2.0,2.0,1.0,193.0,NaN,NaN,-37.85800,144.9005,6380.0
8413,3,12.6,3020.0,3.0,1.0,1.0,555.0,NaN,NaN,-37.79880,144.8220,3755.0
2919,3,13.0,3046.0,3.0,1.0,1.0,265.0,NaN,1995.0,-37.70830,144.9158,8870.0
6043,3,13.3,3020.0,3.0,1.0,2.0,673.0,673.0,1970.0,-37.76230,144.8272,4217.0


In [19]:
# Define scoring function
def score_dataset(X_train, X_valid, y_train, y_valid):
    # Define model
    model = RandomForestRegressor(n_estimators = 10, random_state = 0)
    # Fit model
    model.fit(X_train, y_train)
    # Generate predictions
    y_predictions = model.predict(X_valid)
    
    return mean_absolute_error(y_valid, y_predictions)

In [25]:
# Get missing value columns
list_null = [col for col in X_train.columns if X_train[col].isnull().any()]

#### Drop Columns

In [ ]:
# Drop selected columns
X_train_drop = X_train.drop(list_null, axis = 1)
X_valid_drop = X_valid.drop(list_null, axis = 1)

# Calculate MAE
mae = score_dataset(X_train_drop, X_valid_drop, y_train, y_valid)

# Report
print(f'MAE w/ dropped columns: {mae:.2f}')

MAE w/ dropped columns: 183550.22


#### Simple Imputation

In [28]:
# Define imputer
my_imputer = SimpleImputer()

# Impute values
X_train_simpute = pd.DataFrame(my_imputer.fit_transform(X_train))
X_valid_simpute = pd.DataFrame(my_imputer.transform(X_valid))

# Rename columns
X_train_simpute.columns = X_train.columns
X_valid_simpute.columns = X_valid.columns

# Calculate MAE
mae = score_dataset(X_train_simpute, X_valid_simpute, y_train, y_valid)

# Report
print(f'MAE w/ simple impute: {mae:.2f}')

MAE w/ simple impute: 178166.46


#### Extended Imputation

In [29]:
# Copy data
X_train_plus = X_train.copy()
X_valid_plus = X_valid.copy()

# Generate new columns to record imputation
for column in list_null:
    X_train_plus[column + '_was_missing'] = X_train_plus[column].isnull()
    X_valid_plus[column + '_was_missing'] = X_valid_plus[column].isnull()

# Define imputer
my_imputer = SimpleImputer()
# Impute values
X_train_impute = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
X_valid_impute = pd.DataFrame(my_imputer.transform(X_valid_plus))
# Rename columns
X_train_impute.columns = X_train_plus.columns
X_valid_impute.columns = X_valid_plus.columns

# Calculate MAE
mae = score_dataset(X_train_impute, X_valid_impute, y_train, y_valid)

# Report
print(f'MAE w/ extended impute: {mae:.2f}')

MAE w/ extended impute: 178927.50


From MAE results, you can see that **Simple Imputation** performed better than **Drop Columns** and **Extended Imputation** methods.

## 3. Categorical Variables

A categorical variable takes only a limited number of values.

A survey asking how often you eat breakfast may only have responses: 
- 'Never'
- 'Rarely'
- 'Most days'
- 'Every day'

You will get an error if you try to place these variables into most ML models in Python without preprocessing.

Here, we will discuss 3 approaches for dealing with categorical variables.

### Drop Categorical Variables

The easiest approach to dealing with categorical variables is to remove them from the dataset.

This will only work well if the columns did not contain useful information.

### Ordinal Encoding

Ordinal encoding assigns each unique value to a different integer:
- 'Never' : 0
- 'Rarely' : 1
- 'Most days' : 2
- 'Every day' : 3

This approach assumes some ordering of the categories, shown above by frequency.

For tree-based models like decision trees/random forests, you can epect ordinal encoding to work well with ordinal variables.

**Note**: Training and validation data must contain the same set of categorical values in a given column for ordinal encoding to work.

### One-Hot Encoding

One-hot encoding creates new columns indicating the presence/absence of each possible value in the original data.

| Color  | Red | Yellow | Green |
|--------|-----|--------|-------|
| Red    | 1   | 0      | 0     |
| Red    | 1   | 0      | 0     |
| Yellow | 0   | 1      | 0     |
| Green  | 0   | 0      | 1     |
| Yellow | 0   | 1      | 0     |

One-hot encoding does not assume the ordering of categories, with this kind of data referred to as **nominal variables**.

One-hot encoding generally does not perform well if the categorical variables take on a large number of values.

### Example

In [39]:
# Load data
df = pd.read_csv(MELB)

# Select target
y = df['Price']
X = df.drop(columns = ['Price'])

# Train/test split
X_train_full, X_valid_full, y_train_full, y_valid_full = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state = 0)

# Define columns w/ missing values
list_null = [col for col in X_train_full.columns if X_train_full[col].isnull().any()]
# Drop columns
X_train_full.drop(columns = list_null, inplace = True)
X_valid_full.drop(columns = list_null, inplace = True)

# Select categorical columns w/ low cardinality
list_low = [col for col in X_train_full.columns if 
            X_train_full[col].nunique() < 10 and
            X_train_full[col].dtype == 'object']

# Select numerical columns
list_numerical = [col for col in X_train_full.columns if 
                  X_train_full[col].dtype in ['int64', 'float64']]

# Keep columns
list_columns = list_low + list_numerical
X_train = X_train_full[list_columns].copy()
X_valid = X_valid_full[list_columns].copy()

# Show data
X_train.head()

,Type,Method,Regionname,Rooms,Distance,Postcode,Bedroom2,Bathroom,Landsize,Lattitude,Longtitude,Propertycount
12167,u,S,Southern Metropolitan,1,5.0,3182.0,1.0,1.0,0.0,-37.85984,144.9867,13240.0
6524,h,SA,Western Metropolitan,2,8.0,3016.0,2.0,2.0,193.0,-37.85800,144.9005,6380.0
8413,h,S,Western Metropolitan,3,12.6,3020.0,3.0,1.0,555.0,-37.79880,144.8220,3755.0
2919,u,SP,Northern Metropolitan,3,13.0,3046.0,3.0,1.0,265.0,-37.70830,144.9158,8870.0
6043,h,S,Western Metropolitan,3,13.3,3020.0,3.0,1.0,673.0,-37.76230,144.8272,4217.0


In [40]:
# Get all categorical data
list_cat = [col for col in X_train.columns if X_train[col].dtype == 'object']
print(list_cat)

['Type', 'Method', 'Regionname']


In [43]:
# Define score_dataset function
def score_dataset(X_train, X_valid, y_train, y_valid):

    # Define model
    model = RandomForestRegressor(n_estimators = 100, random_state = 0)
    # Fit model
    model.fit(X_train, y_train)
    # Generate predictions
    y_pred = model.predict(X_valid)
    # Calculate MAE
    mae = mean_absolute_error(y_valid, y_pred)

    return mae

#### Drop Categorical Variables

In [ ]:
# Drop categorical variables
X_train_drop = X_train.select_dtypes(exclude = ['object'])
X_valid_drop = X_valid.select_dtypes(exclude = ['object'])

# Calculate MAE
mae = score_dataset(X_train_drop, X_valid_drop, y_train, y_valid)

# Report
print(F'MAE w/ dropped categorical variables: {mae:.2f}')

MAE w/ dropped categorical variables: 175703.48


#### Ordinal Encoding

For each categorical column, a unique values are assigned a different integer.

There is an additional boost in performance if informed labels are assigned manually.

In [ ]:
# Copy data
X_train_ord = X_train.copy()
X_valid_ord = X_valid.copy()

# Initialise ordinal encoder
ordinal_encoder = OrdinalEncoder()

# Apply ordinal encoding
X_train_ord[list_cat] = ordinal_encoder.fit_transform(X_train[list_cat])
X_valid_ord[list_cat] = ordinal_encoder.transform(X_valid[list_cat])

# Calculae MAE
mae = score_dataset(X_train_ord, X_valid_ord, y_train, y_valid)

# Report
print(F'MAE w/ ordinal encoding: {mae:.2f}')

MAE w/ ordinal encoding: 165936.41


#### One-Hot Encoding

In [51]:
# Initialise One Hot Encoder
onehot_encoder = OneHotEncoder(handle_unknown = 'ignore', sparse_output = False)

# Apply encoding
df_encode_train = pd.DataFrame(onehot_encoder.fit_transform(X_train[list_cat]))
df_encode_valid = pd.DataFrame(onehot_encoder.transform(X_valid[list_cat]))

# Reindex data
df_encode_train.index = X_train.index
df_encode_valid.index = X_valid.index

# Remove old categorical columns
X_train_oh = X_train.drop(columns = list_cat)
X_valid_oh = X_valid.drop(columns = list_cat)

# Add one hot encoded columns to datasets
X_train_encode = pd.concat([X_train_oh, df_encode_train], axis = 1)
X_valid_encode = pd.concat([X_valid_oh, df_encode_valid], axis = 1)

# Assert string type in all columns
X_train_encode.columns = X_train_encode.columns.astype(str)
X_valid_encode.columns = X_valid_encode.columns.astype(str)

# Calculate MAE
mae = score_dataset(X_train_encode, X_valid_encode, y_train, y_valid)

# Report
print(F'MAE w/ one hot encoding: {mae:.2f}')

MAE w/ one hot encoding: 166089.49


There doesn't seem to be any benefit to **Ordinal Encoding** vs **One Hot Encoding**, whereas both are better than the **Drop Categorical Variables** method.

## 4. Pipelines